# Capítulo 1 — El agente que inventa

**Formación Amazon Bedrock AgentCore · octubre 2026 · Caso de uso: Nova Assist (TiendaNova)**

> 🎬 **Escena.** Laura escribe: *«Hola, ¿dónde está mi pedido TN-1001?»*. El asistente responde con mucha seguridad… y se inventa la fecha.

| | |
|---|---|
| **🧠 Componente agéntico** | Modelo e instrucciones: el razonamiento |
| **La necesidad** | Un agente que hable bien y que reconozca lo que no sabe |
| **Lo que construimos** | Agente Strands con *system prompt*, modelo vía inference profile `eu.` y reintentos adaptativos |
| **🏠 En local** | ✅ Funciona: el agente corre en tu notebook (el modelo, eso sí, ya es un servicio remoto). |
| **☁️ Por qué AWS al salir a Internet** | En local usarías una clave de API en un `.env`: con muchas llamadas llegan los 429 (*throttling*) y no controlas dónde se procesan los datos. Bedrock usa IAM sin claves, el inference profile `eu.` reparte la carga entre regiones europeas y los reintentos adaptativos absorben picos. No elimina las cuotas: la aplicación debe saber degradarse. |
| **En el ordenador sería…** | La **ROM**: el modelo trae de fábrica mucho conocimiento general, pero es de solo lectura y no sabe nada de TiendaNova |
| **Requisitos del caso** | RNF-03 (identificarse como IA) |
| **Prerrequisito · duración** | Capítulo 0 · 40 min |
| **⚠️ Dónde se rompe la analogía** | Una ROM devuelve exactamente lo que guarda; el modelo **reconstruye** su conocimiento de forma probabilística. Por eso, cuando no sabe, completa con lo plausible: alucina. |

In [ ]:
# Configuración común: ejecuta siempre esta celda primero
import os
from lab_utils import *

os.environ.setdefault("AWS_REGION", region())
cuaderno()

> 🎮 **¿Llegas tarde o algo se rompió?** La celda siguiente comprueba que tienes todo lo necesario. Si no, descomenta `ponerse_al_dia(1)`: carga la partida guardada al final del capítulo anterior o, si no existe, escribe el código de referencia y te dice qué recursos faltan y en qué capítulo se crean.

In [ ]:
requisitos(1)

## 🧠 La teoría: ¿qué es un agente?

Un agente es un sistema que **persigue un objetivo en un bucle**: razona, actúa, observa el resultado y vuelve a razonar. Un agente **no es un modelo**: es un modelo rodeado de componentes. Este curso los irá encendiendo uno a uno, cada vez que la historia los necesite.

- **El modelo** aporta el razonamiento; **las instrucciones** (*system prompt*) fijan el rol, los objetivos y los límites.
- Sin datos, el modelo solo puede completar con lo plausible: **alucina**.

### El mapa del agente (la columna vertebral del curso)

| Cap. | Componente agéntico | ¿En local? |
|---|---|---|
| 1 | Modelo e instrucciones: el razonamiento | ✅ Funciona |
| 2 | Herramientas y bucle del agente: la acción | ✅ Funciona |
| 3 | Entorno de ejecución: el agente como servicio | ☁️ No aplica |
| 4 | Memoria de trabajo y a corto plazo | ✅ Funciona sin que lo notes |
| 5 | Memoria a largo plazo | ⚠️ Incompleto |
| 6 | Contexto: qué ve el modelo en cada llamada | ✅ Funciona igual en local y en la nube |
| 7 | Interoperabilidad de herramientas (MCP) | ✅ Funciona |
| 8 | Conocimiento: recuperación aumentada (RAG) | ✅ Funciona como prueba |
| 9 | Identidad y delegación | ☁️ No aplica |
| 10 | Guardarraíles y gobernanza | ⚠️ Parcial |
| 11 | Ejecución de código (código como acción) | ⛔ Peligroso |
| 12 | Observabilidad y evaluación | ✅ Funciona a pequeña escala |
| 13 | Mejora continua y experimentación | ☁️ No aplica |
| 14 | Orquestación multi-agente | ✅ Funciona |
| 15 | Ciclo de vida y operación | ☁️ No aplica |

> ✅ funciona en local · ⚠️ incompleto o parcial · ⛔ peligroso · ☁️ solo existe al salir a Internet.
>
> **La teoría manda; AWS entra cuando el agente sale a Internet.** Los capítulos 1 y 2 ocurren en tu máquina. A partir del 3, cada vez que algo deje de funcionar al desplegar, aparecerá la pieza de AWS que lo resuelve.

## 1. Experimento de rotura: un modelo sin reglas

El modelo es una ROM enorme: sabe redactar, razonar y mucho sobre el mundo, pero su conocimiento quedó fijado al entrenarse. No puede saber dónde está el pedido de Laura.

In [ ]:
from strands import Agent
from strands.models import BedrockModel

agente_sin_reglas = Agent(
    model=BedrockModel(model_id=state_get("MODEL_ID"), region_name=region(), temperature=1.0),
    system_prompt="Eres el asistente de TiendaNova.",
)
r = agente_sin_reglas("Hola, ¿dónde está mi pedido TN-1001? Dime la fecha exacta de entrega.")
medir(r, "sin-reglas")

Ejecuta la celda varias veces. ¿Inventa una fecha, un transportista o un estado? Eso es una **alucinación**: el modelo completa con lo plausible lo que no sabe.

## 2. Un agente con reglas (y preparado para producción)

In [ ]:
from botocore.config import Config

# Hilo de producción: reintentos adaptativos ante 429 (throttling) y errores transitorios
reintentos = Config(retries={"mode": "adaptive", "max_attempts": 8})

SYSTEM_PROMPT = """Eres Nova Assist, el asistente de postventa de TiendaNova.
- En tu primer mensaje te presentas como asistente de inteligencia artificial.
- Respondes en el idioma del cliente, con tono cercano y profesional.
- Si no dispones de un dato, lo dices claramente: nunca inventes estados de pedido, fechas ni importes."""

agente = Agent(
    model=BedrockModel(model_id=state_get("MODEL_ID"), region_name=region(), temperature=0.2,
                       boto_client_config=reintentos),
    system_prompt=SYSTEM_PROMPT,
)
r = agente("Hola, ¿dónde está mi pedido TN-1001?")
medir(r, "con-reglas")

**Verificación:** se presenta como IA (art. 50 del AI Act) y reconoce que **no puede consultar el pedido**.

## 3. Hilo de producción: ¿qué pasa con muchas llamadas a la vez?

En producción, cientos de clientes escriben a la vez. Cada cuenta y región tiene **cuotas** de inferencia; si se superan, Bedrock responde con `ThrottlingException` (429). Para responder a eso hay varias capas:

1. **Inference profile `eu.`:** reparte la inferencia entre varias regiones europeas, lo que da más capacidad y resiliencia, sin sacar los datos de la UE.
2. **Reintentos adaptativos** con *backoff*: absorben los picos breves.
3. **Cuotas dimensionadas** (y ampliadas si hace falta) y, si se necesita capacidad garantizada, *provisioned throughput*.
4. **Degradación elegante:** si aun así falla, la aplicación responde algo útil en lugar de caerse.

Lanzamos 10 llamadas simultáneas. Con cuotas normales **probablemente no verás ningún 429**: el objetivo es ver el patrón, no tumbar la cuenta.

In [ ]:
import time
import boto3
from concurrent.futures import ThreadPoolExecutor

rt = boto3.client("bedrock-runtime", region_name=region(), config=reintentos)

def llamada(i):
    t0 = time.perf_counter()
    try:
        rt.converse(modelId=state_get("MODEL_ID"),
                    messages=[{"role": "user", "content": [{"text": f"Responde solo con el número {i}"}]}],
                    inferenceConfig={"maxTokens": 5})
        return "ok", time.perf_counter() - t0
    except rt.exceptions.ThrottlingException:
        return "429", time.perf_counter() - t0

with ThreadPoolExecutor(max_workers=10) as ex:
    resultados = list(ex.map(llamada, range(10)))
print({k: sum(1 for r in resultados if r[0] == k) for k in ("ok", "429")},
      "· latencia máxima:", round(max(r[1] for r in resultados), 2), "s")

In [ ]:
def responder_con_degradacion(pregunta: str) -> str:
    """Patrón de degradación elegante: si la inferencia falla tras los reintentos, no se cae la aplicación."""
    try:
        return str(agente(pregunta))
    except Exception:  # noqa: BLE001  (en producción: registrar la traza y encolar)
        return "Ahora mismo tenemos mucha demanda. Te respondemos en unos minutos por tu canal preferido."

print(responder_con_degradacion("¿Qué horario tenéis?"))

## 4. Arranca el taxímetro

Cada respuesta muestra una línea 🚕 con los *tokens* consumidos. Si fijaste los precios del modelo en el capítulo 0 (`precios(...)`), también verás el coste estimado. Iremos comparando capítulo a capítulo.

In [ ]:
taximetro()

## Para pensar

- Las reglas del *system prompt* reducen la alucinación, pero no aportan datos. ¿Qué pieza falta?
- ¿Dónde debería vivir el requisito del art. 50 (identificarse como IA) para que no dependa de que alguien lo recuerde en cada cambio de prompt?

## 🔎 Lo que acabamos de descubrir

El agente ya no inventa… pero tampoco **sabe** nada: no ve los pedidos. Una ROM no basta; necesita **acceso a datos reales**.

In [ ]:
cuaderno()
taximetro_cerrar(1)
guardar_partida(1)

---
➡️ **Siguiente capítulo:** `02_herramientas.ipynb` — *El agente que no ve tus datos*